### Προετοιμασία δεδομένων

Το πρώτο κελί προετοιμάζει αυτόματα τα εκπαιδευτικά δεδομένα. Εκτέλεσέ το πριν ξεκινήσεις την άσκηση.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import csv
import json
import sqlite3

GROUP = 'core-capstone'
DATA_URLS = {'administrative_zones.gpkg': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/core-capstone/administrative_zones.gpkg', 'roads.gpkg': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/core-capstone/roads.gpkg', 'services.csv': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/core-capstone/services.csv', 'study_area.gpkg': 'https://raw.githubusercontent.com/pkazser/geoskills/main/downloads/python/data/core-capstone/study_area.gpkg'}
SCHEMA = {'administrative_zones.gpkg': {'kind': 'gpkg', 'layers': ['administrative_zones']}, 'roads.gpkg': {'kind': 'gpkg', 'layers': ['roads']}, 'services.csv': {'kind': 'csv', 'fields': ['service_id', 'name', 'category', 'longitude', 'latitude', 'status']}, 'study_area.gpkg': {'kind': 'gpkg', 'layers': ['study_area']}}

def valid_cached(path, spec):
    if not path.exists() or path.stat().st_size == 0:
        return False
    try:
        if spec["kind"] == "csv":
            with path.open(encoding="utf-8-sig", newline="") as handle:
                reader = csv.DictReader(handle)
                return set(spec["fields"]).issubset(set(reader.fieldnames or []))
        if spec["kind"] == "geojson":
            document = json.loads(path.read_text(encoding="utf-8"))
            if document.get("type") != "FeatureCollection" or not isinstance(document.get("features"), list):
                return False
            return all(isinstance(feature, dict) and isinstance(feature.get("properties"), dict) and set(spec["fields"]).issubset(feature["properties"]) for feature in document["features"])
        if spec["kind"] == "gpkg":
            if path.read_bytes()[:16] != b"SQLite format 3\x00":
                return False
            with sqlite3.connect(path) as connection:
                if connection.execute("PRAGMA integrity_check").fetchone()[0] != "ok":
                    return False
                layers = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents")}
            return set(spec["layers"]).issubset(layers)
    except Exception:
        return False
    return False

def valid_dataset_root(base):
    folder = base / GROUP
    return folder.is_dir() and all(valid_cached(folder / filename, spec) for filename, spec in SCHEMA.items())

candidates = (Path("downloads/python/data"), Path("../../data"), Path("data"))
DATA_ROOT = next((candidate for candidate in candidates if valid_dataset_root(candidate)), None)

if DATA_ROOT is None:
    DATA_ROOT = Path("data")
    target_root = DATA_ROOT / GROUP
    target_root.mkdir(parents=True, exist_ok=True)
    failures = []
    for filename, url in DATA_URLS.items():
        target = target_root / filename
        if valid_cached(target, SCHEMA[filename]):
            print("[OK] Already available:", target)
            continue
        if target.exists():
            target.unlink()
        partial = target.with_name(target.name + ".part")
        if partial.exists():
            partial.unlink()
        try:
            with urlopen(url, timeout=30) as response:
                payload = response.read()
            if not payload:
                raise RuntimeError("empty response")
            partial.write_bytes(payload)
            partial.replace(target)
            if not valid_cached(target, SCHEMA[filename]):
                raise RuntimeError("minimum validation failed")
            print("[OK] Downloaded and validated:", target)
        except Exception as exc:
            if partial.exists():
                partial.unlink()
            if target.exists():
                target.unlink()
            failures.append(filename + ": " + str(exc))
    if failures or not valid_dataset_root(DATA_ROOT):
        raise RuntimeError("Could not prepare dataset " + GROUP + ": " + "; ".join(failures))
    print("[OK] Dataset root:", DATA_ROOT.resolve())
else:
    print("[OK] Using validated dataset root:", (DATA_ROOT / GROUP).resolve())


# CORE CAPSTONE — Public Services Coverage

Ακολούθησε τη ροή Κατανοώ → Ελέγχω → Επεξεργάζομαι → Επαληθεύω → Εξάγω. Το notebook έχει σκόπιμα κενά.

In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

root = DATA_ROOT / "core-capstone"
# TODO: έλεγξε ότι τα τέσσερα input paths υπάρχουν


## 1. Audit

CHECKPOINT: φόρτωσε τα layers και το CSV. Κατέγραψε rows, columns, CRS, geometry types και bounds. Hint 1: ξεκίνα με `gpd.read_file` και `pd.read_csv`. Hint 2: χρησιμοποίησε `.shape`, `.crs` και `.total_bounds`. Hint 3: το CSV δεν έχει geometry ακόμη. Για το συγκεκριμένο input αναμένεται η ακολουθία 11 raw rows → 10 μετά το duplicate handling → 8 με αριθμημένες coordinates → 7 εντός study area.

In [ ]:
# TODO: study = ...
# TODO: zones = ...
# TODO: roads = ...
# TODO: services_raw = ...


## 2. Cleaning

DECISION: ποια records θα κρατήσεις και γιατί; Μετέτρεψε longitude/latitude σε numeric, έλεγξε duplicates και missing values. EXPLAIN: γιατί είναι σωστό `errors='coerce'`;

Hint 1: `pd.to_numeric`. Hint 2: `drop_duplicates` και `dropna`. Hint 3: κράτησε counts πριν/μετά. Για αυτό το input κατέγραψε 11 raw rows, 10 μετά το duplicate `S08`, 8 μετά τις μη έγκυρες coordinates και 7 μετά τον χωρικό έλεγχο.

In [ ]:
# TODO: numeric coordinates
# TODO: create a GeoDataFrame with points and EPSG:4326
# TODO: remove points outside study area
# TODO: print raw, numeric and cleaned counts


## 3. Common spatial framework

Χρησιμοποίησε projected CRS κατάλληλο για την Ελλάδα. CHECKPOINT: εξήγησε σε markdown γιατί δεν κάνεις buffer σε degrees.

In [ ]:
target_crs = 'EPSG:2100'
# TODO: reproject study, zones, roads and cleaned services


## 4. Explore and spatial join

Βρες τη ζώνη κάθε υπηρεσίας. QA: πρέπει να έχεις 7 services και κανένα missing zone_id. Εξήγησε τι προσθέτει το spatial join. Το `zone_id` υπάρχει μόνο αφού ολοκληρωθεί η αντιστοίχιση.

In [ ]:
# TODO: services_with_zones = gpd.sjoin(...)
# TODO: inspect counts and unmatched rows


## 5. Analysis

DECISION: active Health/Emergency services, 500 m buffer. Χρησιμοποίησε buffer, clip και overlay. Hint 1: φίλτρο με `&` και `isin`. Hint 2: κάνε dissolve πριν από overlay. Hint 3: δημιούργησε πρώτα `zones_p["zone_area_m2"] = zones_p.geometry.area` και έλεγξε ότι covered_m2 ≤ zone_area_m2. Το τελικό summary πρέπει να κρατά και τις τρεις zones, με 0 όπου δεν υπάρχει intersection.

In [ ]:
# TODO: active_essential = ...
# TODO: service_buffers = ...
# TODO: clipped_roads = ...
# TODO: coverage_by_zone = ...


## 6. QA

Συμπλήρωσε assertions για CRS, rows, geometry validity, bounds, unique ids, unmatched zones και positive areas. Πρόσθεσε έναν χάρτη για visual QA.

In [ ]:
# TODO: assertions
# TODO: plot zones, roads, services and buffers


## 7. Export

EXPLAIN: ποια layers χρειάζεται να παραδώσει άλλος αναλυτής; Εξήγαγε τα πέντε απαιτούμενα layers του GeoPackage, summary.csv με και τις τρεις zones, final_map.png και short_findings.md στον φάκελο `output/core-capstone/`. Ξαναδιάβασε τα outputs και έλεγξε rows, CRS, geometries και συμφωνία summary με όλες τις zones.

In [ ]:
# TODO: create output directory
# TODO: export logical layers and summary
# TODO: re-read at least one exported layer
